# Interactive fixed effects

This notebook shows the frozen call surface only. It does not sample, and it does not name a prior family. The prior family is not frozen.

The untreated counterfactual is the additive unit intercept plus the additive time intercept plus the loading-factor product. `max_factors` is an upper bound. `shrinkage_state` is one scale draw per factor, not a selected rank.


In [ ]:
import pandas as pd
import xarray as xr

from causalpy.experiments.interactive_fixed_effects import (
    InteractiveFixedEffectsSyntheticControl,
    untreated_counterfactual,
)

frame = pd.DataFrame(
    {
        "donor": [0.0, 1.0, 2.0, 3.0],
        "treated": [1.0, 1.2, 5.0, 5.4],
    },
    index=[0, 1, 2, 3],
)
experiment = InteractiveFixedEffectsSyntheticControl(
    frame,
    treatment_time=2,
    control_units=["donor"],
    treated_units=["treated"],
    max_factors=1,
)
unit = xr.DataArray(
    [[[0.2]]],
    dims=("chain", "draw", "treated_units"),
    coords={"treated_units": ["treated"]},
)
time = xr.DataArray(
    [[[0.0, 0.1, 0.2, 0.3]]],
    dims=("chain", "draw", "obs_ind"),
    coords={"obs_ind": [0, 1, 2, 3]},
)
loadings = xr.DataArray(
    [[[[1.0]]]],
    dims=("chain", "draw", "treated_units", "factors"),
    coords={"treated_units": ["treated"], "factors": ["f0"]},
)
factors = xr.DataArray(
    [[[[0.5], [0.5], [0.5], [0.5]]]],
    dims=("chain", "draw", "obs_ind", "factors"),
    coords={"obs_ind": [0, 1, 2, 3], "factors": ["f0"]},
)
scales = xr.DataArray(
    [[[0.05]]],
    dims=("chain", "draw", "factors"),
    coords={"factors": ["f0"]},
)
counterfactual = untreated_counterfactual(unit, time, loadings, factors)
result = experiment.compose_result(
    unit_intercept=unit,
    time_intercept=time,
    loadings=loadings,
    factors=factors,
    shrinkage_state=scales,
)
result.predictions_post, result.shrinkage_state, counterfactual